# FHIR Basics — Part 1: Searching

This is the first notebook in a series that explains, from first principles,
how this application talks to its FHIR R4 server (the NHS North West
Genomics IG, https://nw-gmsa.github.io/en/). It's aimed at someone who knows
Python but hasn't worked with FHIR before.

**This notebook covers *searching* only** — `GET` requests that find
resources. It does not cover creating/updating resources (`POST`/`PUT`),
which is a separate, much smaller part of this app (see `/order/new`'s
"Send to ESB" button in `fhir_client.py`) and will get its own notebook
later in the series.

## The one-sentence version

A FHIR search is an HTTP `GET` against a resource type's endpoint, with
query-string parameters narrowing the results:

```
GET [base]/[ResourceType]?param1=value1&param2=value2
```

`[base]` for this app is whatever `FHIR_BASE_URL` is configured to (see
`CLAUDE.md`) — in this deployment, that's:

```
https://192.168.1.62/healthconnect/cdr/fhir/r4
```

Every example URL below uses that base. If you're pointed at a different
server, just swap the host.

## Every search returns a Bundle

You never get a bare list back — you always get a `Bundle` resource of
type `searchset`, shaped roughly like:

```json
{
  "resourceType": "Bundle",
  "type": "searchset",
  "total": 2,
  "entry": [
    {"resource": {"resourceType": "Patient", "id": "123", ...}, "search": {"mode": "match"}},
    {"resource": {"resourceType": "Organization", "id": "456", ...}, "search": {"mode": "include"}}
  ],
  "link": [
    {"relation": "self", "url": "..."},
    {"relation": "next", "url": "..."}
  ]
}
```

Three things worth knowing up front, because they show up repeatedly in
this app's own code (`fhir_client.py`):

- **`entry[].search.mode`** tells you whether a resource is an actual
  match for your query (`"match"`) or was only pulled in as extra context
  via `_include`/`_revinclude` (`"include"`) — see the `_include` section
  near the end of this notebook. Some servers don't tag this reliably, so
  this app is deliberately cautious about trusting it (see
  `CLAUDE.md`'s "413s on unfiltered system-wide searches" and the ctDNA
  section for real examples of that quirk).
- **`Bundle.link[rel=next]`** is how you page through more than one page
  of results — follow it to get the next batch. `fhir_client.py`'s
  `_search_all()` does this automatically, capped at `max_pages` (default
  10).
- **`_count`** controls page size (how many resources come back per
  request), not the total.

## Authentication

This deployment uses HTTP Basic auth — the same username/password you'd
log into this app with at `/login`. In **Postman**, use the **Authorization**
tab → type **Basic Auth** → enter your username/password, rather than
building the `Authorization: Basic ...` header by hand. Every example
below assumes that's set.

You'll also want an `Accept: application/fhir+json` header — most FHIR
servers default to JSON anyway, but it's good practice to be explicit.

## A note on how this notebook is organised

For each kind of search, you'll get:

1. A short explanation of the search parameter(s) involved.
2. A Python cell that builds the exact URL using this app's own
   conventions (so you can see how the query string is assembled).
3. A **"Postman" markdown block** with the finished URL ready to paste
   into Postman's address bar (or `curl`).

None of the code cells below call the FHIR server directly — they only
*build URLs* — so this notebook is safe to run without server access.
There's an optional "live" cell at the very end if you want to try a real
call with `requests`.

In [40]:
from urllib.parse import urlencode
from pathlib import Path
import os

# Load the repo's .env file (FHIR_BASE_URL, FHIR_USER, FHIR_PASSWORD, ...)
# into os.environ — a bare os.environ.get() only ever sees variables the
# shell/kernel process already had exported, never a .env file sitting on
# disk, so without this step FHIR_USER/FHIR_PASSWORD look "missing" even
# when they're right there in .env. This notebook lives in docs/notebooks/,
# two levels below the repo root where .env actually is, so search upward
# for it rather than assuming the current working directory.
try:
    from dotenv import load_dotenv
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        env_path = candidate / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            print(f"Loaded environment variables from {env_path}")
            break
    else:
        print("No .env file found in this directory or any parent — relying on "
              "already-exported environment variables, if any.")
except ImportError:
    print("python-dotenv isn't installed, so .env won't be loaded automatically. "
          "Run `pip install python-dotenv`, or export FHIR_BASE_URL/FHIR_USER/"
          "FHIR_PASSWORD yourself before starting Jupyter.")

# Matches CLAUDE.md's example deployment. Overridden by .env's FHIR_BASE_URL
# (or the FHIR_BASE_URL env var directly) if either is set.
BASE_URL = os.environ.get("FHIR_BASE_URL", "https://192.168.1.62/healthconnect/cdr/fhir/r4")


def fhir_search_url(resource_type, params):
    """Build a FHIR search URL the same shape fhir_client.py's _get() does.

    `params` is a dict; a value that's a list produces one repeated query
    parameter per item (FHIR's AND semantics — see the date-range section
    below for why that distinction matters). `doseq=True` is what makes
    urlencode do that.
    """
    query = urlencode(params, doseq=True)
    return f"{BASE_URL}/{resource_type}?{query}"


print(fhir_search_url("Patient", {"_count": 5}))

Loaded environment variables from /Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/julius/.env
https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?_count=5


## Suggested Python tooling

This app deliberately talks to FHIR with plain `requests` calls rather
than a FHIR client library, specifically so the raw REST calls stay
visible (see `fhir_client.py`'s module docstring). That's a good default
for learning FHIR too — but if you're prototyping something bigger, these
are worth knowing about:

- **`requests`** — already a dependency of this app (`requirements.txt`);
  everything in this notebook uses it, or could.
- **`urllib.parse.urlencode(..., doseq=True)`** — used above; handles both
  single values and repeated parameters (needed for `ge`/`le` date ranges)
  correctly, so you don't hand-build query strings.
- **`fhir.resources`** (PyPI) — Pydantic models for every FHIR R4 resource
  type, if you want typed parsing/validation of what comes back instead of
  working with raw `dict`s. Not used in this app on purpose (see above),
  but genuinely useful for other projects.
- **`python-dotenv`** — the setup cell above actually uses this one (not
  just a suggestion): it loads the repo's `.env` file
  (`FHIR_BASE_URL`/`FHIR_USER`/`FHIR_PASSWORD`) into `os.environ` for you.
  Note this is notebook-only tooling — the Flask app itself
  (`app.py`/`fhir_client.py`) never reads `.env`, only real environment
  variables, so `python-dotenv` isn't in the app's own `requirements.txt`.
  Run `pip install python-dotenv` if the setup cell reports it's missing.

## 1. Search Patient by name

`Patient?name=` is a fuzzy match against `Patient.name` (given name,
family name, or both) — good for "find someone by what they're called"
but not reliable enough to use for a definite match, since names can
repeat or be spelled inconsistently. This app only uses it as a *fallback*
after trying the NHS number first (see "Patient matching" in
`CLAUDE.md`).

In [41]:
url = fhir_search_url("Patient", {"name": "Smith", "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?name=Smith&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?name=Smith&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

## 2. Search Patient by date of birth

`Patient?birthdate=` matches against `Patient.birthDate`. Plain
`birthdate=1980-05-12` is an exact-date match; the same `ge`/`le` prefixes
covered in more depth in the ServiceRequest date section below also work
here (e.g. `birthdate=ge1980-01-01&birthdate=le1980-12-31` for "born
sometime in 1980").

In [42]:
url = fhir_search_url("Patient", {"birthdate": "1980-05-12", "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?birthdate=1980-05-12&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?birthdate=1980-05-12&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

## 3. Search Patient by NHS Number

NHS Number lives on `Patient.identifier`, so it's found via the generic
`identifier` search parameter, not a dedicated `nhs-number` one. This
IG's patients carry it under a known system URI
(`FhirClient.NHS_NUMBER_SYSTEM` =
`https://fhir.nhs.uk/Id/nhs-number`), but this app deliberately searches
by **bare value** (`identifier=9000000009`) rather than
`system|value` — see `search_patients()`'s own comment in
`fhir_client.py`: FHIR's `identifier` search matches by value alone when
no `system|` prefix is given, which is more forgiving of a server that
tags the identifier with a slightly different system URI than expected.

If you *do* know the system and want an exact, unambiguous match, the
`system|value` form is shown below too — useful when a bare value search
returns more than you expect (e.g. it happens to also match a CHI number
or an internal ID with the same digits).

In [43]:
nhs_number = "9000000009"  # example NHS number

# What this app actually sends (bare value — matches any identifier system):
url_bare = fhir_search_url("Patient", {"identifier": nhs_number, "_count": 20})
print("Bare value match:      ", url_bare)

# The more explicit, system-qualified equivalent:
NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
url_qualified = fhir_search_url("Patient", {"identifier": f"{NHS_NUMBER_SYSTEM}|{nhs_number}", "_count": 20})
print("System-qualified match:", url_qualified)

Bare value match:       https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=9000000009&_count=20
System-qualified match: https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https%3A%2F%2Ffhir.nhs.uk%2FId%2Fnhs-number%7C9000000009&_count=20


**Postman (bare value, what this app sends):**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=9000000009&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

**Postman (system-qualified, for an unambiguous match):**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https%3A%2F%2Ffhir.nhs.uk%2FId%2Fnhs-number%7C9000000009&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

(Postman URL-decodes the address bar for you as you type/paste, so pasting
the readable form `identifier=https://fhir.nhs.uk/Id/nhs-number|9000000009`
works too — it'll show the encoded version once sent.)

## 4. Search Patient by Medical Record Number (MRN)

An MRN is also a `Patient.identifier` entry, but typed as HL7 v2-0203 code
`"MR"` (`FhirClient.MEDICAL_RECORD_NUMBER_TYPE`) rather than carrying its
own dedicated system URI — a patient can have *several* MRNs, one per
assigning organisation (see `medical_record_numbers()` in
`fhir_client.py`).

FHIR's plain `identifier` search parameter matches on **value**, not on
the identifier's `type` code — there's no `identifier-type=MR` shortcut in
base FHIR search. So searching for an MRN looks exactly like searching for
any other identifier value; the difference is just *which* value you
already know (an MRN rather than an NHS number).

In [44]:
mrn = "M1234567"  # example medical record number

url = fhir_search_url("Patient", {"identifier": mrn, "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=M1234567&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=M1234567&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

If the MRN happens to collide with some other identifier's value on a
different patient (unlikely, but possible on a bare-value search), you'd
need to fetch the candidate(s) and check `identifier[].type.coding[].code
== "MR"` yourself client-side — this app's `medical_record_numbers()`
does exactly that filtering after the fact, rather than trying to push it
into the search.

## 5. Search Organization by ODS code

An NHS ODS (Organisation Data Service) code is, again, just an
`Organization.identifier` value, so it's found the same
`identifier=` way. This app's `search_organizations(ods_code=...)`
searches by **bare value** here too — the exact system URI a real ODS
code is tagged with on this server is unconfirmed (see `CLAUDE.md`, "Things
that are unverified", item 8), so matching on value alone is the safer
default rather than guessing a system URI that might not match.

In [45]:
ods_code = "RW3"  # example ODS trust code

url = fhir_search_url("Organization", {"identifier": ods_code, "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/Organization?identifier=RW3&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Organization?identifier=RW3&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

Note: this app deliberately never does an *unfiltered* `Organization`
search from user input (no name/ODS code given → it returns `[]` without
querying at all) — an unscoped search is what causes the 413s documented
in `CLAUDE.md`'s "413s on unfiltered system-wide searches" section. Always
give `identifier` (or `name`) a real value.

## 6. Search ServiceRequest by intent

`ServiceRequest.intent` distinguishes *who's looking at the order*:
`order`/`original-order` (the placer/requesting side) vs `filler-order`
(the filler/lab side). This app's Work Orders (`/work-orders`) and Test
Orders (`/test-orders`) screens are built entirely around this one
parameter.

**The important gotcha**: to match *either* of two intents, join them with
a comma **inside one `intent=` parameter** — that's FHIR's OR semantics.
Two *separate* `intent=` parameters would instead mean AND, which no
single `ServiceRequest` (it only ever has one `intent` value) could ever
satisfy, so a repeated-parameter query would come back empty. This is
called out directly in `_active_orders_with_intent()`'s docstring in
`fhir_client.py` because it's an easy mistake to make — repeated
parameters *do* mean AND elsewhere (see the date-range section below), so
the comma-vs-repeat distinction is genuinely parameter-specific, not a
general rule.

In [46]:
# Filler-side orders only (single value):
url_filler = fhir_search_url("ServiceRequest", {"intent": "filler-order", "_count": 20})
print("Filler-order only:      ", url_filler)

# Placer-side orders — two intents, comma-joined for OR:
url_placer = fhir_search_url("ServiceRequest", {"intent": "order,original-order", "_count": 20})
print("Order OR original-order:", url_placer)

Filler-order only:       https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?intent=filler-order&_count=20
Order OR original-order: https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?intent=order%2Coriginal-order&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?intent=filler-order&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?intent=order,original-order&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

## 7. Search ServiceRequest by performer

`ServiceRequest.performer` is a reference to who's meant to *carry out*
the requested test — a `Practitioner`, `PractitionerRole`, `Organization`,
etc. The `performer` search parameter matches by reference, so you
generally need a resource id (`Organization/123`) rather than a name.

If you only know a *name* (e.g. "find every order performed by Dr Smith"),
FHIR supports **chained search** — `performer:Practitioner.name=` reaches
through the reference into the target resource's own search parameters in
one request, instead of two round trips (look up the Practitioner's id
first, then search `ServiceRequest?performer=Practitioner/<id>`).

In [47]:
performer_org_id = "456"  # example Organization id on this server

# By reference (you already know the id):
url_by_ref = fhir_search_url("ServiceRequest", {"performer": f"Organization/{performer_org_id}", "_count": 20})
print("By reference:  ", url_by_ref)

# By chained search on the performer's name (no id needed up front):
url_chained = fhir_search_url("ServiceRequest", {"performer:Practitioner.name": "Smith", "_count": 20})
print("Chained search:", url_chained)

By reference:   https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?performer=Organization%2F456&_count=20
Chained search: https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?performer%3APractitioner.name=Smith&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?performer=Organization/456&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?performer:Practitioner.name=Smith&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

(Chained search support varies by server — if it comes back empty
unexpectedly, fall back to the two-step "search Practitioner by name,
then search ServiceRequest by performer=<id>" approach and compare.)

## 8. Search ServiceRequest by order numbers (placer/filler)

An order number — placer (assigned by the requesting system) or filler
(assigned by the lab system) — is, once again, just a
`ServiceRequest.identifier` value, distinguished by an HL7 v2-0203 type
code (`"PLAC"`/`"FILL"`, `FhirClient.PLACER_IDENTIFIER_TYPE`/
`FILLER_IDENTIFIER_TYPE`). Exactly like the MRN case above, base FHIR
`identifier` search matches on value only, not on the type code, so a
placer number and a filler number are searched exactly the same way —
this app's `find_orders_by_identifier()` deliberately takes advantage of
that: the caller doesn't need to know upfront which *kind* of order
number they have, since both kinds live in the same searchable field.

In [48]:
order_number = "LE20260815ab12cd"  # example placer order number

url = fhir_search_url("ServiceRequest", {"identifier": order_number, "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?identifier=LE20260815ab12cd&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?identifier=LE20260815ab12cd&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

The same pattern works for `DiagnosticReport?identifier=` too (e.g. an
iGene report identifier) — this app's search screen tries both
`ServiceRequest` and `DiagnosticReport` for whatever number you type in,
since you can't tell which one it is just by looking at it.

## 9. Date parameters: `ge` and `le` on ServiceRequest

FHIR date search parameters accept a **prefix** in front of the date/time
value that changes the comparison:

| Prefix | Meaning |
|---|---|
| `eq` (default) | equal to |
| `ge` | greater than or equal to (on/after) |
| `le` | less than or equal to (on/before) |
| `gt` | strictly after |
| `lt` | strictly before |

For a date **range** — the common "between these two dates" case — you
combine `ge` and `le` as **two separate query parameters with the same
name**. Unlike the `intent` comma-joining above, repeated parameters here
mean **AND**: `authored=ge2026-08-01&authored=le2026-08-31` means "on or
after 1 Aug *and* on or before 31 Aug" — a single `ServiceRequest.authored`
value is checked against both conditions at once, so AND is exactly what
you want. (Contrast this directly with the `intent` case above, where AND
would have been useless — the parameter-repetition rule genuinely depends
on the field, not a blanket convention.)

This is exactly the shape `_active_orders_with_intent()` and
`orders_in_range()` build in `fhir_client.py`:
`{"authored": [f"ge{start}", f"le{end}"]}` — a Python list value, which
`urlencode(..., doseq=True)` (used by `fhir_search_url()` above) turns
into two repeated `authored=` parameters automatically.

In [49]:
start = "2026-08-01"
end = "2026-08-20"

url = fhir_search_url("ServiceRequest", {
    "authored": [f"ge{start}", f"le{end}"],
    "_count": 100,
})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?authored=ge2026-08-01&authored=le2026-08-20&_count=100


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?authored=ge2026-08-01&authored=le2026-08-20&_count=100
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

## 10. Search Specimen by specimen identifier

Specimens carry their own identifiers too —
`FhirClient.SPECIMEN_IDENTIFIER_SYSTEM`
(`https://fhir.nwgenomics.nhs.uk/iGene/SpecimenIdentifier`) is the
IG-specific system this app knows about for the iGene specimen ID. Same
`identifier=` parameter, same bare-value-vs-system-qualified choice as the
NHS number example above.

In [50]:
specimen_id = "SP0012345"  # example specimen identifier

url_bare = fhir_search_url("Specimen", {"identifier": specimen_id, "_count": 20})
print("Bare value match:      ", url_bare)

SPECIMEN_IDENTIFIER_SYSTEM = "https://fhir.nwgenomics.nhs.uk/iGene/SpecimenIdentifier"
url_qualified = fhir_search_url("Specimen", {"identifier": f"{SPECIMEN_IDENTIFIER_SYSTEM}|{specimen_id}", "_count": 20})
print("System-qualified match:", url_qualified)

Bare value match:       https://192.168.1.62/healthconnect/cdr/fhir/r4/Specimen?identifier=SP0012345&_count=20
System-qualified match: https://192.168.1.62/healthconnect/cdr/fhir/r4/Specimen?identifier=https%3A%2F%2Ffhir.nwgenomics.nhs.uk%2FiGene%2FSpecimenIdentifier%7CSP0012345&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Specimen?identifier=SP0012345&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

## 11. Search ServiceRequest or Specimen by patient id

Both `ServiceRequest` and `Specimen` support the `patient` search
parameter — a reference-type parameter that matches resources whose
subject is a given `Patient`. `ServiceRequest.subject` and
`Specimen.subject` are both aliased by this parameter (FHIR defines
`patient` as shorthand for "search `subject`, but only when it points at
a Patient"), so you use the exact same parameter name for both resource
types even though the underlying field is `subject` either way.

You can pass either the bare id (`123`) or the full relative reference
(`Patient/123`) — both are accepted, and this app's own code uses the
bare-id form (see `lab_orders_for_patient()`'s `{"patient": patient_id,
...}` in `fhir_client.py`).

In [51]:
patient_id = "123"

url_orders = fhir_search_url("ServiceRequest", {"patient": patient_id, "_count": 50})
url_specimens = fhir_search_url("Specimen", {"patient": patient_id, "_count": 50})
print("Orders for patient:   ", url_orders)
print("Specimens for patient:", url_specimens)

Orders for patient:    https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?patient=123&_count=50
Specimens for patient: https://192.168.1.62/healthconnect/cdr/fhir/r4/Specimen?patient=123&_count=50


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?patient=123&_count=50
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Specimen?patient=123&_count=50
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

### A step further: `_include` to get related resources in one call

If you're about to look at a patient's orders *and* who requested them
*and* the specimen for each, you can pull all of that back in a single
query instead of one request per resource, using `_include`:

```
GET .../ServiceRequest?patient=123&_include=ServiceRequest:specimen&_include=ServiceRequest:requester&_count=50
```

The extra resources come back in the same `Bundle`, tagged
`"search": {"mode": "include"}` (as opposed to `"match"` for the actual
`ServiceRequest`s) — see the Bundle explanation at the top of this
notebook. This is exactly what `lab_orders_for_patient()` does in
`fhir_client.py` (see "Reference resolution" in `CLAUDE.md`), and it's
the difference between one request and an N+1 fan-out of individual GETs
for every related resource.

## Optional: try a real call

If you have network access to a real FHIR server and a valid
username/password, this cell will actually run a search against it and
show you a real `Bundle`. It's guarded so the notebook doesn't fail if
you don't — everything above this point works without it.

It searches for NHS number **9737383206** — one of the published test
patients from the NW GM-SA IG's own testing page
(https://nw-gmsa.github.io/en/testing.html#genomics-test-patients), so
this is safe to run against a real conformant server without needing to
know a real patient's details first.

Uses the same `FHIR_USER`/`FHIR_PASSWORD` env vars `fhir_client.py` itself
falls back to for directly-constructed clients (e.g. `scripts/
fix_organization_names.py`) — see `CLAUDE.md`'s Authentication section.
The running app no longer reads these (it uses per-user `/login`
credentials instead), but they're still the existing convention for a
standalone script/notebook like this one, so this reuses them rather than
inventing a new, notebook-only env var name.

**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=9737383206&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

In [52]:
import requests
import json as _json

FHIR_USER = os.environ.get("FHIR_USER")
FHIR_PASSWORD = os.environ.get("FHIR_PASSWORD")

# Published NW GM-SA test patient — see
# https://nw-gmsa.github.io/en/testing.html#genomics-test-patients
TEST_NHS_NUMBER = "9737383206"

if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
else:
    resp = requests.get(
        fhir_search_url("Patient", {"identifier": TEST_NHS_NUMBER, "_count": 20}),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    bundle = resp.json()
    matches = [
        entry["resource"] for entry in bundle.get("entry", [])
        if entry.get("resource", {}).get("resourceType") == "Patient"
    ]
    print(f"{len(matches)} Patient match(es) for NHS number {TEST_NHS_NUMBER}")
    for patient in matches:
        name = (patient.get("name") or [{}])[0]
        display_name = " ".join([*name.get("given", []), name.get("family", "")]).strip()
        print(f" - id={patient['id']}  name={display_name or '(no name)'}  birthDate={patient.get('birthDate')}")
    print()
    print(_json.dumps(bundle, indent=2)[:2000])

1 Patient match(es) for NHS number 9737383206
 - id=184459  name=Ned LIVERPOOL  birthDate=1942-06-18

{
  "resourceType": "Bundle",
  "id": "d77e66fe-ee18-4054-bbba-17001afa6501",
  "type": "searchset",
  "timestamp": "2026-08-20T10:21:30Z",
  "total": 1,
  "link": [
    {
      "relation": "self",
      "url": "https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?_count=20&identifier=9737383206"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://192.168.1.62/healthconnect/cdr/fhir/r4/Patient/184459",
      "resource": {
        "resourceType": "Patient",
        "address": [
          {
            "id": "bcIsv",
            "line": [
              "20 Forthlin Road",
              "LIVERPOOL",
              "null"
            ],
            "period": {
              "start": "2023-09-08"
            },
            "postalCode": "L18 9TN",
            "use": "home"
          }
        ],
        "birthDate": "1942-06-18",
        "gender": "male",
        "generalPractitioner":

/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/julius/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### And the same for Specimen search by identifier

The IG's testing page publishes NHS numbers for its test patients, but no
fixed specimen identifiers to search on — there's nothing to hardcode a
Postman example against ahead of time the way section 10 above does with
a made-up `SP0012345`. So this cell finds a *real* one instead: it takes
the test patient matched above (`Specimen?patient=<id>`), pulls the
identifier off their first specimen, then re-searches
`Specimen?identifier=` with that real value — the same two-step chain
covered in section 11's `_include` note, just done as two separate calls
instead of one combined query. The Postman-ready URL for that second call
is printed as part of the output, since (unlike the NHS number above)
there's no value fixed enough to write into this markdown cell directly.

In [ ]:
if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
elif not matches:
    print("No Patient match from the cell above to look up specimens for.")
else:
    test_patient_id = matches[0]["id"]

    resp = requests.get(
        fhir_search_url("Specimen", {"patient": test_patient_id, "_count": 20}),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    specimens = [
        entry["resource"] for entry in resp.json().get("entry", [])
        if entry.get("resource", {}).get("resourceType") == "Specimen"
    ]
    print(f"{len(specimens)} Specimen(s) for patient {test_patient_id}")

    specimen_identifiers = [
        ident["value"] for s in specimens for ident in s.get("identifier", []) if ident.get("value")
    ]
    if not specimen_identifiers:
        print("None of this patient's specimens carry an identifier — nothing to search on.")
    else:
        specimen_id_value = specimen_identifiers[0]
        identifier_url = fhir_search_url("Specimen", {"identifier": specimen_id_value, "_count": 20})
        print(f"Using specimen identifier: {specimen_id_value}")
        print()
        print("Postman-ready URL:")
        print(f"  GET {identifier_url}")
        print("  Accept: application/fhir+json")
        print("  Authorization: Basic Auth (your username/password)")

        resp2 = requests.get(
            identifier_url,
            auth=(FHIR_USER, FHIR_PASSWORD),
            headers={"Accept": "application/fhir+json"},
            verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
            timeout=30,
        )
        resp2.raise_for_status()
        bundle2 = resp2.json()
        print()
        print(f"Bundle.total for that search: {bundle2.get('total')}")
        print(_json.dumps(bundle2, indent=2)[:2000])

## What's next

This notebook covered read-only search — the FHIR verbs and parameters
this app uses to *find* things. **Part 2**
(`02-work-orders-worked-example.ipynb`) is published and builds directly
on it: a full worked example of finding a laboratory's current work
orders via `Task`, retrieving the underlying order/specimen/patient,
handling the case where the patient is a fetus or baby with no NHS number
yet (via `RelatedPerson`), and the `Task.status` lifecycle a lab is
expected to drive an order through.

Topics neither notebook covers yet, for a future Part 3 onward:

- Category-coded searches and status/status-reason filtering — the
  try-categorized-then-fall-back pattern this app uses throughout
  (`ServiceRequest.category`/`DiagnosticReport.category` — see
  "Category codes come from the IG, not guesses" in `CLAUDE.md`).
- Pagination (`Bundle.link[rel=next]`, `_count`) and why an unscoped
  system-wide search can 413 a real server — the organisation-scoped
  batching pattern this app had to adopt (see "413s on unfiltered
  system-wide searches" in `CLAUDE.md`).
- Writing data for real — building and sending a FHIR message `Bundle`
  (this app's order-creation screen, `/order/new`), and the `Task.status`
  update Part 2 deliberately stopped short of sending.